# ARES Persona Red-Teaming Configuration Generator

This notebook creates custom configuration files for running persona-based adversarial attacks against language models using the ARES framework.

## Purpose
Generate YAML configuration files that define:
- **Target model settings**: The model being tested for vulnerabilities
- **Attacker model settings**: The model generating adversarial prompts
- **Attack parameters**: Tree depth, branching factor, number of personas
- **Strategy configurations**: For persona generation, tree exploration, and output handling.

## Workflow
1. Load a base configuration template
2. Set attack parameters (depth, personas, strategies)
3. Configure target and attacker models
4. Generate and save the configuration file
5. Use the config with `ares evaluate <path>`

## Step 1: Import Required Libraries

In [ ]:
import yaml
import os

## Step 2: Load Base Configuration Template

Load an existing configuration file as a starting point. This template contains the basic structure needed for persona red-teaming attacks.

In [ ]:
# Load the base configuration template
original_config = yaml.safe_load(open('example_configs/plugins/ares_persona/assets/persona_example_ollama_small.yaml'))

# Create a copy to modify without affecting the original template
modified_config = original_config.copy()

## Step 3: Configure Attack Parameters

These parameters control the behavior and complexity of the red-teaming attack.

### Parameter Descriptions:

- **`branching_factor`**: Number of conversation branches to explore at each tree node
  - Higher values = more diverse attack attempts but slower execution
  - Recommended: 2-3 for balanced exploration vs. speed

- **`N_CHARS`**: Number of distinct personas to use in the attack
  - Each persona has unique characteristics and conversation strategies
  - More personas = more diverse attack angles but higher computational cost
  - Recommended: Start with 3-6, increase for more comprehensive testing

- **`DEPTH`**: Maximum number of conversation turns in the attack tree
  - Higher depth = longer conversations, potentially more sophisticated attacks
  - Typical range: 5-15 turns
  - Recommended: Start with 5-8, increase gradually

- **`RUN`**: Run identifier for organizing multiple experimental runs
  - Increment this when running multiple experiments with different parameters

- **`STRAT_SET`**: Number of attack strategies assigned to each character
  - Strategies define how personas manipulate conversations (e.g., emotional appeal, authority)
  - Higher values = more complex persona behavior

- **`NAME`**: Configuration name for organizing output files and results
  - Change this for different experimental setups

In [ ]:
# Attack tree branching factor
branching_factor = 3

# Number of personas/characters
N_CHARS = 6

# Maximum conversation depth
DEPTH = 10

# Experiment run identifier
RUN = 3

# Strategies per character
STRAT_SET = 2

# Configuration name
NAME = 'custom_config'

## Step 4: Initialize Output Configuration

In [ ]:
# List to store paths of generated configuration files
config_paths = []

# List to store bash commands for running experiments (if needed)
bash_commands = []

# Output folder where generated prompts and results will be saved
output_folder = 'assets/' + NAME

## Step 5: Configure Target Model (Model Being Tested)

### Target Model Configuration:

- **`target_as_detector`**: Determines how the target model is used
  - `False` (default): Target is a standard LLM being tested for jailbreaks
  - `True`: Target is a safety classifier that outputs safe/unsafe judgments

- **`model_path`**: Hugging Face model identifier or local path
  - Examples: `"meta-llama/Llama-3.1-8B-Instruct"`, `"mistralai/Mistral-7B-Instruct-v0.2"`
  - This is the model you want to test for vulnerabilities

In [ ]:
# Set to True if your model is a safety detector/classifier
target_as_detector = False

# Hugging Face model path for the target model
# Change this to test different models
model_path = "meta-llama/Llama-3.1-8B-Instruct"

# Extract model name from path for configuration naming
name = model_path.split("/")[-1]

## Step 6: Configure Hugging Face Target Model Connector

This configuration defines how to load and interact with the target model using Hugging Face Transformers.

### Key Configuration Options:

- **`dtype`**: Data type for model weights
  - `"bfloat16"`: Memory efficient, good for most GPUs
  - `"float16"`: Alternative for older GPUs
  - `"float32"`: Full precision, requires more memory

- **`padding_side`**: Must be `"left"` for batch generation

- **`max_new_tokens`**: Maximum tokens to generate per response
  - Adjust based on expected response length
  - Higher values = longer responses but slower generation

- **`device`**: Computation device
  - `"cuda"`: Use GPU (strongly recommended)
  - `"cpu"`: Use CPU (much slower)

- **`system_prompt`**: Initial instruction given to the model
  - Standard mode: "You are a helpful assistant."
  - Detector mode: Empty string

In [ ]:
config_huggingface_connector = {
    # Connector type: Specifies the ARES connector class to use
    "type": "ares.connectors.huggingface.HuggingFaceConnector",
    
    # Connector name: Identifier for this model configuration
    "name": f"{name}",
    
    # Model configuration: Parameters for loading the model
    "model_config": {
        # Path to pretrained model on Hugging Face Hub or local path
        "pretrained_model_name_or_path": model_path,
        
        # Data type for model weights (bfloat16 for memory efficiency)
        "dtype": "bfloat16"
    },
    
    # Tokenizer configuration: Parameters for the tokenizer
    "tokenizer_config": {
        # Path to tokenizer (usually same as model path)
        "pretrained_model_name_or_path": model_path,
        
        # Padding side: "left" is required for batch generation
        "padding_side": "left"
    },
    
    # Generation parameters: Controls how the model generates responses
    "generate_kwargs": {
        # Chat template configuration: Formats conversations for the model
        "chat_template": {
            # Return PyTorch tensors
            "return_tensors": "pt",
            
            # Thinking mode: Some models support chain-of-thought reasoning
            # Set to False for standard generation
            "thinking": False,
            "enable_thinking": False,
            
            # Return dictionary format for easier parsing
            "return_dict": True,
            
            # Add generation prompt: Prepares model for response generation
            "add_generation_prompt": True,
            
            # Decoding parameters: Controls output formatting
            'decode_params': {
                # Skip special tokens (e.g., <|endoftext|>, <pad>) in output
                'skip_special_tokens': True,
            }
        },
        
        # Generation parameters: Controls response length
        "generate_params": {"max_new_tokens": 512}
    },
    
    # Random seed for reproducibility
    "seed": 42,
    
    # Device: "cuda" for GPU, "cpu" for CPU (GPU strongly recommended)
    "device": "cuda",
    
    # System prompt: Initial instruction given to the model
    "system_prompt": {
        'role': 'system', 
        'content': 'You are a helpful assistant.' if not target_as_detector else ''
    }
}

## Step 7: Configure Attacker Model (Generates Adversarial Prompts)

This configuration defines the model that will generate attack prompts. Using LiteLLM connector allows integration with various API providers.

### Supported Endpoint Types:
- **`ollama`**: Local Ollama server
- **`huggingface`**: Hugging Face Inference API
- **`Other`**: Check `litellm` documentation

### Model Format Examples:
- Ollama: `"ollama/qwen3:8b"`, `"ollama/llama3:8b"`

## Ollama configuration example

In [ ]:
config_ollama_attacker = {
    # Connector type: LiteLLM supports multiple providers
    "type": "ares_litellm.LiteLLMConnector",
    
    # Connector name: Identifier for the attacker model
    "name": "qwen3-on-ollama",
    
    # Endpoint type: Specifies the API provider
    "endpoint-type": "ollama",
    
    # Model identifier: Format depends on endpoint-type
    "model": "ollama/qwen3:8b"
}

# Store target model name for later use
fixed_target_name = config_huggingface_connector['name']

## Step 8: Configure Persona Red-Teaming Strategy

This section modifies the loaded configuration with custom parameters.

### Configuration Details:

- **`local_target`**: Whether the target model runs locally (True) or via API (False)

- **`target_as_detector`**: Whether target is a safety classifier
  - `False`: Standard LLM
  - `True`: Binary safe/unsafe classifier

- **`safe_keyword`**: Keyword in detector output indicating safe response
  - Only used when `target_as_detector=True`, define the keyword that indicates a safe response for the used model
  - `'N'`: No jailbreak detected
  - `'Y'`: Jailbreak detector

- **`use_fixed_strategies`**: Strategy assignment mode
  - `True`: Each character has fixed strategies (user can pre define)
  - `False`: Sample new strategies per character (more diverse)

- **`strategies_path`**: YAML file containing available attack strategies
  - Strategies include: emotional appeal, authority, urgency, etc.

- **`n_strategies_per_character`**: Number of strategies each persona uses
  - Higher values = more complex persona behavior

In [ ]:
# Target model configuration
modified_config['persona-redteaming']['strategy']['persona-strategy']['target_model_config']['local_target'] = target_as_detector
modified_config['persona-redteaming']['strategy']['persona-strategy']['target_model_config']['target_as_detector'] = False

# Safe keyword for detector mode (only used if target_as_detector=True)
modified_config['persona-redteaming']['strategy']['persona-strategy']['target_model_config']['safe_keyword'] = 'N'

# Tree parameters: Controls the attack tree structure
modified_config['persona-redteaming']['strategy']['persona-strategy']['tree_params']['depth'] = DEPTH

# Persona model configuration: Number of distinct personas
modified_config['persona-redteaming']['strategy']['persona-strategy']['persona_model_config']['number_of_characters'] = N_CHARS

# Strategy configuration: Defines how personas manipulate conversations
modified_config['persona-redteaming']['strategy']['persona-strategy']['strategy_config'] = {
    # Use fixed strategies for all characters or sample new ones
    'use_fixed_strategies': False,
    
    # Path to strategies YAML file
    'strategies_path': f'assets/strategies.yaml',
    
    # Number of strategies per character
    'n_strategies_per_character': STRAT_SET
}

## Step 9: Assign Model Connectors to Configuration

In [ ]:
# Set the attacker model connector (generates adversarial prompts)
target_model_config = modified_config['persona-redteaming']['strategy']['persona-strategy']['attacker_model_config']['connector'] = config_ollama_attacker

# Get reference to target model configuration section
target_model_config = modified_config['persona-redteaming']['strategy']['persona-strategy']['target_model_config']

# Store target model name
target_model_name = fixed_target_name

# Set output folder for generated prompts and conversation logs
modified_config['persona-redteaming']['strategy']['persona-strategy']['prompts_folder'] = output_folder

# Set branching factor (number of branches per tree node)
modified_config['persona-redteaming']['strategy']['persona-strategy']['tree_params']['branching_factor'] = branching_factor

# Set the target model connector (model being tested)
target_model_config['connector'] = config_huggingface_connector

# Update the target model configuration in the main config
modified_config['persona-redteaming']['strategy']['persona-strategy']['target_model_config'] = target_model_config

## Step 10: Save Configuration File

Generate the final YAML configuration file that can be used with the ARES CLI.

In [ ]:
# Construct output path for the configuration file
yaml_filename = f'example_configs/plugins/ares_persona/{output_folder}'
path = yaml_filename + "/persona_example" + target_model_name + ".yaml"

# Add path to list of generated configs (useful for batch processing)
config_paths.append(path)

# Create output directory if it doesn't exist
os.makedirs(yaml_filename, exist_ok=True)

# Write the modified configuration to a YAML file
with open(yaml_filename + "/persona_example" + target_model_name + ".yaml", 'w') as f:
    yaml.dump(modified_config, f)
    print(f"Config saved to {yaml_filename}/persona_example{target_model_name}.yaml")

## Next Steps for Practitioners

After running this notebook, you can:

### 1. Review the Generated Configuration
Verify the settings in the generated YAML file to ensure they match your requirements.

### 2. Run the Red-Teaming Attack
Use the ARES CLI to execute the attack:
```bash
ares evaluate <path_to_generated_config.yaml>
```

### 3. Monitor Progress
Check the output folder specified above for:
- Generated conversation logs
- Persona definitions
- Attack success metrics

### 4. Analyze Results
Use the `result_reader.ipynb` notebook to analyze attack outcomes.
